# Teacher labelling — reconstruct invoice tables with the 35B MoE

**Phase two, step 1 of the distillation pipeline.** The served 35B MoE (`qwen3.6-35b-a3b-fp8`, vLLM, OpenAI-compatible, on the L40) reads each invoice image and emits the table as HTML. Those `(image, html)` pairs become the training labels for the 8B student in `finetune-and-serve.ipynb`.

The invoices are confidential and **cannot leave the network** — every call in this notebook goes to `localhost`. Nothing here touches an external service.

**The images are noisy.** Besides the printed table they carry handwritten notes, stamps, and a QR/barcode with its own printed reference string. The prompt's whole job is to make the model reconstruct **only the printed data table** and ignore the rest.

Why chain-of-thought here: deciding that a handwritten scrawl or the text beside a QR is *not* a table cell is a spatial-reasoning call, and CoT measurably helps VLMs on that. The teacher runs offline to mint labels, so its extra tokens cost nothing that matters. We keep the reasoning in a `<reasoning>` block and parse only the final `<table>`.

## Config

Everything you might change lives in this one cell.

In [ ]:
import sys, base64, json, time, mimetypes
from pathlib import Path
import requests

# Repo root on sys.path so `src` imports work from the notebook.
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# --- vLLM endpoint (the 35B teacher) ---------------------------------
BASE_URL   = 'http://localhost:8000/v1'   # vLLM OpenAI-compatible server
MODEL_NAME = 'qwen3.6-35b-a3b-fp8'         # must match `--served-model-name`
API_KEY    = 'EMPTY'                        # vLLM ignores it; header must exist

# --- data locations --------------------------------------------------
IMAGES_DIR = ROOT / 'data' / 'invoices'    # your ~20 invoice images
OUT_DIR    = ROOT / 'data' / 'teacher'     # per-image html + manifest
OUT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST   = OUT_DIR / 'labels.jsonl'      # -> input to notebook 2

# --- decoding --------------------------------------------------------
USE_COT        = True    # reasoning block before the HTML (recommended)
TEMPERATURE    = 0.0     # deterministic labels
MAX_TOKENS     = 4096    # full invoice tables + reasoning
REQUEST_TIMEOUT = 300

IMAGE_GLOB = ('*.png', '*.jpg', '*.jpeg', '*.webp', '*.tif', '*.tiff')
print('root       :', ROOT)
print('endpoint   :', BASE_URL, '->', MODEL_NAME)
print('images dir :', IMAGES_DIR)

## The prompt

One system message that fixes the role and the ignore-list, and one instruction that asks for brief reasoning then the final HTML. The ignore-list is explicit and exhaustive — vagueness here is what lets a QR reference number leak in as a phantom cell.

In [ ]:
TEACHER_SYSTEM = (
    'You are an expert document-table reconstruction engine. You convert a '
    'scanned or photographed business document into a faithful HTML '
    'representation of ONLY its printed data table.'
)

# The exhaustive ignore-list. Shared with the student's concise prompt in
# notebook 2 so teacher and student agree on what counts as 'the table'.
IGNORE_LIST = (
    'Do NOT transcribe or represent any of the following, even if they sit '
    'inside or overlap the table:\n'
    '  - handwritten text, signatures, ticks, or margin notes;\n'
    '  - stamps, seals, watermarks, or logos;\n'
    '  - QR codes and barcodes;\n'
    '  - any printed text that encodes, captions, or labels a QR/barcode '
    '(e.g. a reference or tracking string printed beside or beneath it).'
)

RECON_RULES = (
    'Reconstruct the printed data table as HTML. Preserve every rowspan, '
    'colspan, merged cell, header hierarchy (use <th> for header cells), '
    'and the natural reading order. Transcribe the printed text of each '
    'cell exactly as shown. Leave a cell empty if it is blank in the image.'
)

COT_INSTRUCTION = (
    RECON_RULES + '\n\n' + IGNORE_LIST + '\n\n'
    'Work in two steps:\n'
    '1. Inside a <reasoning>...</reasoning> block, briefly locate the '
    'printed table, state its row/column layout and any merged cells, and '
    'name anything you are excluding per the ignore-list.\n'
    '2. After the reasoning block, output the final answer as raw HTML that '
    'begins with <table> and ends with </table>. No markdown fences, no '
    'text after the </table>.'
)

DIRECT_INSTRUCTION = (
    RECON_RULES + '\n\n' + IGNORE_LIST + '\n\n'
    'Output only the raw HTML starting with <table> and ending with '
    '</table>. No markdown fences, no explanation.'
)

INSTRUCTION = COT_INSTRUCTION if USE_COT else DIRECT_INSTRUCTION
print(INSTRUCTION)

## Endpoint helpers

`clean_prediction` from `src/model/prompts.py` already strips markdown fences and trims to the outermost `<table>...</table>` — so it lifts the final HTML straight out of a CoT response (the `<reasoning>` prose has no `<table>` tag to confuse it). We reuse it rather than re-inventing a parser.

In [ ]:
from src.model.prompts import clean_prediction


def image_data_url(path: Path) -> str:
    """Base64 data URL. vLLM accepts these inline so nothing hits disk
    or the network beyond localhost."""
    mime = mimetypes.guess_type(str(path))[0] or 'image/png'
    b64 = base64.b64encode(path.read_bytes()).decode()
    return f'data:{mime};base64,{b64}'


def reconstruct(path: Path, instruction: str = INSTRUCTION) -> dict:
    """Call the teacher on one image. Returns {raw, html, reasoning}."""
    payload = {
        'model': MODEL_NAME,
        'temperature': TEMPERATURE,
        'max_tokens': MAX_TOKENS,
        'messages': [
            {'role': 'system', 'content': TEACHER_SYSTEM},
            {'role': 'user', 'content': [
                {'type': 'image_url',
                 'image_url': {'url': image_data_url(path)}},
                {'type': 'text', 'text': instruction},
            ]},
        ],
    }
    r = requests.post(
        f'{BASE_URL}/chat/completions',
        headers={'Authorization': f'Bearer {API_KEY}'},
        json=payload, timeout=REQUEST_TIMEOUT,
    )
    r.raise_for_status()
    raw = r.json()['choices'][0]['message']['content']
    reasoning = ''
    if '<reasoning>' in raw and '</reasoning>' in raw:
        reasoning = raw.split('<reasoning>', 1)[1].split('</reasoning>', 1)[0].strip()
    return {'raw': raw, 'html': clean_prediction(raw), 'reasoning': reasoning}

## Smoke test on one image

Run this before the batch. Confirm the server answers, the reasoning is sane, and the HTML parses.

In [ ]:
images = sorted(p for pat in IMAGE_GLOB for p in IMAGES_DIR.glob(pat))
assert images, f'no images found in {IMAGES_DIR}'
print(len(images), 'images')

out = reconstruct(images[0])
print('--- reasoning ---\n', out['reasoning'][:800])
print('\n--- html (first 800 chars) ---\n', out['html'][:800])
assert out['html'].startswith('<table'), 'no <table> parsed -- inspect out["raw"]'

In [ ]:
# Eyeball the parsed table rendered next to nothing -- just to see it is a table.
from IPython.display import HTML, Image as IPyImage, display
display(IPyImage(filename=str(images[0]), width=420))
display(HTML(out['html']))

## Batch — label every invoice

Writes one `.html` per image and appends to `labels.jsonl`. Idempotent: re-running skips images already in the manifest, so a mid-batch server hiccup does not cost the labels already earned.

In [ ]:
def load_done(manifest: Path) -> set:
    if not manifest.exists():
        return set()
    return {json.loads(l)['uid'] for l in manifest.read_text().splitlines() if l.strip()}

done = load_done(MANIFEST)
print(len(done), 'already labelled; ', len(images) - len(done), 'to go')

with MANIFEST.open('a') as mf:
    for i, path in enumerate(images, 1):
        uid = path.stem
        if uid in done:
            continue
        t0 = time.time()
        try:
            out = reconstruct(path)
        except Exception as e:
            print(f'  [{i}/{len(images)}] {uid}: FAILED {e}')
            continue
        if not out['html'].startswith('<table'):
            print(f'  [{i}/{len(images)}] {uid}: no table parsed -- skipped, inspect manually')
            (OUT_DIR / f'{uid}.raw.txt').write_text(out['raw'])
            continue
        (OUT_DIR / f'{uid}.html').write_text(out['html'])
        rec = {'uid': uid, 'image_path': str(path), 'html': out['html'],
               'model': MODEL_NAME, 'reasoning': out['reasoning']}
        mf.write(json.dumps(rec, ensure_ascii=False) + '\n')
        mf.flush()
        print(f'  [{i}/{len(images)}] {uid}: ok ({time.time()-t0:.1f}s, '
              f'{len(out["html"])} chars)')
print('done ->', MANIFEST)

## Human review — the only quality gate you have

With no ground truth, **your eyes are the eval set.** These labels train the student, so a hallucinated cell here becomes a learned error there. Skim every one: image on the left, reconstructed table on the right. For invoices specifically, a cheap automatic sniff test is totals reconciliation — `sum(line items) + tax ≈ total`; a mismatch flags a bad label without any labelled reference.

In [ ]:
from IPython.display import HTML, Image as IPyImage, display

records = [json.loads(l) for l in MANIFEST.read_text().splitlines() if l.strip()]
print(len(records), 'labelled tables\n')
for rec in records:
    print('=' * 80, '\n', rec['uid'])
    display(IPyImage(filename=rec['image_path'], width=420))
    display(HTML(rec['html']))

---
**Next:** open `finetune-and-serve.ipynb`. It reads this `labels.jsonl`, fine-tunes the 8B student on it, and serves the adapter with vLLM.

> ⚠️ These are *teacher* labels, not ground truth. Correct the ones you flagged before training — a 35B model still miscounts merged cells on a hard invoice, and the student will faithfully learn whatever you feed it.